# CardioIA - Fase 2 - Parte 2
## Classificador de risco a partir de frases de sintomas

Aqui eu treino um modelo que lê uma frase de paciente e responde **alto risco** ou
**baixo risco**, simulando a triagem que decide quem é atendido primeiro.

Caminho: TF-IDF para transformar texto em números, depois um classificador do
scikit-learn. Avalio com acurácia, matriz de confusão e validação cruzada.

**Aviso:** exercício acadêmico com base sintética. Não é ferramenta de triagem.

### 1. Carregando a base rotulada

`dados/frases_risco.csv` tem 80 frases escritas pelo grupo: 40 de alto risco e 40 de
baixo risco. Além de `frase` e `situacao`, existe uma coluna `grupo_semantico` — a
próxima seção explica para que ela serve.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

# Reaproveito a normalização da Parte 1: mesma limpeza de acento e maiúscula.
from extracao_sintomas import normalizar

pd.set_option("display.max_colwidth", 80)

df = pd.read_csv(RAIZ / "dados" / "frases_risco.csv")
print(f"{len(df)} frases | {df.grupo_semantico.nunique()} grupos semânticos")
df.situacao.value_counts()

80 frases | 40 grupos semânticos


situacao
alto risco     40
baixo risco    40
Name: count, dtype: int64

In [2]:
df.head(6)

,frase,situacao,grupo_semantico
0,Estou com uma dor forte no peito que desce pelo braço esquerdo e não passa d...,alto risco,dor_toracica_irradiada
1,"A dor no meu peito espalhou para o queixo e para o ombro esquerdo, começou f...",alto risco,dor_toracica_irradiada
2,dor no peito irradiando pro braco esquerdo com formigamento na mao,alto risco,dor_toracica_irradiada
3,"Sinto um aperto muito forte no peito mesmo parado no sofá, sem ter feito esf...",alto risco,dor_toracica_em_repouso
4,A dor no peito apareceu enquanto eu dormia e continua mesmo depois de eu lev...,alto risco,dor_toracica_em_repouso
5,De repente minha fala embolou e o lado direito do rosto caiu.,alto risco,sinais_neurologicos_subitos


In [3]:
# Tamanho dos grupos: de 1 a 3 frases cada.
df.grupo_semantico.value_counts().head(8)

grupo_semantico
dor_toracica_irradiada             3
sinais_neurologicos_subitos        3
dispneia_subita                    3
dor_musculoesqueletica_toracica    3
refluxo_azia                       3
sono_ruim                          3
dor_toracica_em_repouso            2
sincope                            2
Name: count, dtype: int64

### 2. Por que dividir por grupo, e não por frase

Estas três frases são variações da mesma ideia clínica:

- "Estou com uma dor forte no peito que desce pelo braço esquerdo..."
- "A dor no meu peito espalhou para o queixo e para o ombro esquerdo..."
- "dor no peito irradiando pro braco esquerdo com formigamento na mao"

Se uma delas cair no treino e outra no teste, o modelo não está generalizando: ele está
reconhecendo uma frase que já viu quase igual. A acurácia sobe sem que o modelo tenha
ficado melhor.

A regra geral é: **dividir pela origem da amostra, não pela amostra.** A coluna
`grupo_semantico` existe para tornar essa divisão possível.

Uso `StratifiedGroupKFold`, que mantém grupos inteiros de um lado só **e** equilibra as
duas classes entre treino e teste.

In [4]:
X, y, grupos = df["frase"], df["situacao"], df["grupo_semantico"]

separador = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
idx_treino, idx_teste = next(separador.split(X, y, groups=grupos))

X_treino, X_teste = X.iloc[idx_treino], X.iloc[idx_teste]
y_treino, y_teste = y.iloc[idx_treino], y.iloc[idx_teste]

print(f"treino: {len(X_treino)} frases | teste: {len(X_teste)} frases")
print("teste por classe:", y_teste.value_counts().to_dict())

treino: 60 frases | teste: 20 frases
teste por classe: {'alto risco': 10, 'baixo risco': 10}


In [5]:
# Conferência obrigatória: nenhum grupo pode aparecer nos dois lados.
vazando = set(grupos.iloc[idx_treino]) & set(grupos.iloc[idx_teste])
print("grupos presentes em treino E teste:", len(vazando))
assert len(vazando) == 0, "Vazamento entre treino e teste!"

grupos presentes em treino E teste: 0


### 3. TF-IDF: transformando frase em número

O modelo não lê texto, lê número. O TF-IDF monta uma tabela onde cada coluna é uma
palavra (ou par de palavras) e cada linha é uma frase.

O valor combina duas ideias:

- **TF** (frequência): quantas vezes o termo aparece naquela frase;
- **IDF** (frequência inversa): quanto mais raro o termo é no conjunto todo, mais peso
  ele recebe. Palavras que aparecem em quase toda frase — "de", "que", "eu" — acabam
  valendo pouco sozinhas, e é isso que a gente quer.

Duas escolhas que fiz:

- `preprocessor=normalizar` reaproveita a função da Parte 1, então "coracao" e "coração"
  viram o mesmo termo aqui também;
- `ngram_range=(1, 2)` inclui pares de palavras. Isso importa muito neste problema:
  "em repouso" e "sem esforço" só significam alguma coisa como par.

In [6]:
vetor = TfidfVectorizer(preprocessor=normalizar, ngram_range=(1, 2), sublinear_tf=True)
matriz = vetor.fit_transform(X_treino)
print(f"matriz: {matriz.shape[0]} frases x {matriz.shape[1]} termos")

matriz: 60 frases x 858 termos


In [7]:
# Como fica uma frase depois do TF-IDF: só os termos com peso maior que zero.
exemplo = X_treino.iloc[0]
linha = pd.Series(matriz[0].toarray()[0], index=vetor.get_feature_names_out())
print(exemplo, "\n")
linha[linha > 0].sort_values(ascending=False).head(10)

De repente minha fala embolou e o lado direito do rosto caiu. 



caiu             0.249183
do rosto         0.249183
direito do       0.249183
rosto caiu       0.249183
repente minha    0.249183
embolou lado     0.249183
embolou          0.249183
fala embolou     0.249183
rosto            0.249183
minha fala       0.249183
dtype: float64

### 4. Modelo 1: Regressão Logística

Escolhi Regressão Logística como modelo principal por dois motivos práticos: ela lida
bem com muitas colunas e poucas linhas (é o nosso caso: 80 frases, mais de mil termos),
e dá para inspecionar o peso de cada termo depois — o que permite explicar a decisão.

`class_weight="balanced"` evita que o modelo prefira a classe maior. Aqui as classes
estão equilibradas, mas deixo ligado porque em triagem real nunca estão.

In [8]:
def montar_pipeline(classificador):
    """TF-IDF e classificador no mesmo objeto.

    Usar Pipeline não é enfeite: garante que o TF-IDF seja ajustado SÓ com os dados de
    treino dentro de cada divisão da validação cruzada. Se eu ajustasse o vetorizador na
    base inteira antes de dividir, o vocabulário do teste vazaria para o treino.
    """
    return Pipeline([
        ("tfidf", TfidfVectorizer(preprocessor=normalizar, ngram_range=(1, 2), sublinear_tf=True)),
        ("clf", classificador),
    ])

modelo_lr = montar_pipeline(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
)
modelo_lr.fit(X_treino, y_treino)
pred_lr = modelo_lr.predict(X_teste)

print(f"Acurácia no teste: {accuracy_score(y_teste, pred_lr):.1%}")

Acurácia no teste: 90.0%


### 5. Matriz de confusão

A acurácia sozinha esconde **que tipo** de erro o modelo comete. Em triagem os dois
erros não custam a mesma coisa:

- **falso negativo** — paciente grave classificado como baixo risco: pode morrer na fila;
- **falso positivo** — paciente leve classificado como alto risco: ocupa recurso à toa.

Um sistema de triagem tem que errar para o lado do falso positivo.

*(A matriz está em tabela, não em gráfico, porque o projeto restringe as dependências a
pandas e scikit-learn — matplotlib ficaria fora da regra.)*

In [9]:
matriz_conf = pd.crosstab(y_teste, pred_lr, rownames=["real"], colnames=["previsto"])
matriz_conf

previsto,alto risco,baixo risco
real,,
alto risco,10,0
baixo risco,2,8


In [10]:
print(classification_report(y_teste, pred_lr, digits=3))

              precision    recall  f1-score   support

  alto risco      0.833     1.000     0.909        10
 baixo risco      1.000     0.800     0.889        10

    accuracy                          0.900        20
   macro avg      0.917     0.900     0.899        20
weighted avg      0.917     0.900     0.899        20



In [11]:
# Traduzindo a matriz para os termos da triagem.
fn = matriz_conf.loc["alto risco", "baixo risco"] if "baixo risco" in matriz_conf.columns else 0
fp = matriz_conf.loc["baixo risco", "alto risco"] if "alto risco" in matriz_conf.columns else 0
print(f"Falsos negativos (grave tratado como leve) : {fn}")
print(f"Falsos positivos (leve tratado como grave) : {fp}")

Falsos negativos (grave tratado como leve) : 0
Falsos positivos (leve tratado como grave) : 2


### 6. Modelo 2: Árvore de Decisão, para comparar

O enunciado cita Decision Tree como alternativa. Treino a mesma coisa com árvore para
ver se a escolha do modelo muda o resultado.

In [12]:
modelo_dt = montar_pipeline(
    DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42)
)
modelo_dt.fit(X_treino, y_treino)
pred_dt = modelo_dt.predict(X_teste)

print(f"Árvore de Decisão - acurácia: {accuracy_score(y_teste, pred_dt):.1%}")
pd.crosstab(y_teste, pred_dt, rownames=["real"], colnames=["previsto"])

Árvore de Decisão - acurácia: 70.0%


previsto,alto risco,baixo risco
real,,
alto risco,8,2
baixo risco,4,6


### 7. Comparando com um baseline burro

Antes de comemorar qualquer acurácia, preciso saber quanto vale o palpite mais idiota
possível. Baseline: se a frase contém "peito", "coracao", "falta de ar" ou "braco",
chuta alto risco; senão, baixo risco. Zero aprendizado, só busca de palavra.

Se o modelo treinado não bater isso com folga, ele não está entregando nada.

In [13]:
palavras_chave = ["peito", "coracao", "falta de ar", "braco"]
pred_baseline = [
    "alto risco" if any(p in normalizar(f) for p in palavras_chave) else "baixo risco"
    for f in X_teste
]

print(f"Baseline por palavra-chave : {accuracy_score(y_teste, pred_baseline):.1%}")
print(f"Regressão Logística        : {accuracy_score(y_teste, pred_lr):.1%}")
print(f"Árvore de Decisão          : {accuracy_score(y_teste, pred_dt):.1%}")

Baseline por palavra-chave : 60.0%
Regressão Logística        : 90.0%
Árvore de Decisão          : 70.0%


### 8. Validação cruzada: o número em que eu realmente acredito

A acurácia da seção 4 vem de **uma única divisão** com 20 frases de teste. Com uma base
desse tamanho, trocar a semente aleatória muda o resultado vários pontos. Reportar só
aquele número seria enganoso.

A validação cruzada roda cinco divisões diferentes e mostra a média e o desvio. É esse
o número que vai para o README.

In [14]:
cv_grupo = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

cv_lr = cross_val_score(montar_pipeline(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    X, y, groups=grupos, cv=cv_grupo)
cv_dt = cross_val_score(montar_pipeline(
    DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42)),
    X, y, groups=grupos, cv=cv_grupo)

print(f"Regressão Logística : {cv_lr.mean():.1%} +/- {cv_lr.std():.1%}   folds: {np.round(cv_lr, 3)}")
print(f"Árvore de Decisão   : {cv_dt.mean():.1%} +/- {cv_dt.std():.1%}   folds: {np.round(cv_dt, 3)}")

Regressão Logística : 77.5% +/- 5.0%   folds: [0.812 0.688 0.812 0.75  0.812]
Árvore de Decisão   : 63.7% +/- 8.3%   folds: [0.688 0.625 0.75  0.625 0.5  ]


### 9. Medindo o vazamento que a coluna de grupo evita

Dá para quantificar o problema. Abaixo eu rodo a mesma validação cruzada duas vezes:

- **por grupo** (`StratifiedGroupKFold`): variações da mesma ideia ficam do mesmo lado;
- **por frase** (`StratifiedKFold`): o jeito comum, que deixa frases quase iguais
  caírem em lados opostos.

A diferença é o tamanho da mentira que o vazamento conta.

In [15]:
cv_vazando = cross_val_score(montar_pipeline(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    X, y, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))

print(f"CV dividindo por GRUPO (correto) : {cv_lr.mean():.1%}")
print(f"CV dividindo por FRASE (vazando) : {cv_vazando.mean():.1%}")
print(f"Inflação causada pelo vazamento  : +{(cv_vazando.mean() - cv_lr.mean()) * 100:.1f} pontos percentuais")

CV dividindo por GRUPO (correto) : 77.5%
CV dividindo por FRASE (vazando) : 83.8%
Inflação causada pelo vazamento  : +6.2 pontos percentuais


### 10. Onde o modelo errou

Olhar as frases erradas ensina mais do que a acurácia.

In [16]:
erros = pd.DataFrame({
    "frase": X_teste,
    "real": y_teste,
    "previsto": pred_lr,
    "grupo": grupos.iloc[idx_teste],
})
erros = erros[erros.real != erros.previsto]

if len(erros) == 0:
    print("Nenhum erro nesta divisão.")
else:
    for _, l in erros.iterrows():
        print(f"\nreal: {l['real']:12} previsto: {l['previsto']:12} [{l['grupo']}]")
        print(f"  {l['frase']}")


real: baixo risco  previsto: alto risco   [dor_musculoesqueletica_toracica]
  Fiz supino na academia e o músculo do peito está doendo ao levantar o braço.

real: baixo risco  previsto: alto risco   [dispneia_pos_exercicio]
  Corri cinco quilômetros e fiquei com falta de ar, mas normalizou em poucos minutos parado.


**Os dois erros foram na mesma direção, e é a direção certa.**

Nenhum falso negativo: o modelo não deixou passar nenhuma frase grave. Os dois erros
foram frases leves classificadas como alto risco — "dor no peito" de academia e "falta
de ar" depois de corrida. São justamente os casos difíceis que eu coloquei de propósito
na base, com vocabulário de alto risco em contexto benigno.

Em triagem, errar para cima é o erro tolerável. Mas atenção: **isso não foi projetado,
foi sorte desta divisão.** Em outra divisão da validação cruzada pode aparecer falso
negativo. Com 20 frases de teste, nenhuma conclusão sobre tipo de erro é estável.

### 11. Quais termos o modelo usa para decidir

Um dos motivos de eu ter escolhido Regressão Logística: dá para abrir e ver o peso de
cada termo. Coeficiente positivo empurra para alto risco, negativo para baixo risco.

In [17]:
tfidf_ajustado = modelo_lr.named_steps["tfidf"]
coefs = pd.Series(
    modelo_lr.named_steps["clf"].coef_[0],
    index=tfidf_ajustado.get_feature_names_out(),
)
# A classe positiva é a segunda em ordem alfabética: 'baixo risco'.
print("classes na ordem do modelo:", modelo_lr.named_steps["clf"].classes_)

classes na ordem do modelo: ['alto risco' 'baixo risco']


In [18]:
print("Termos que mais puxam para BAIXO RISCO:")
print(coefs.sort_values(ascending=False).head(12).round(3).to_string())
print("\nTermos que mais puxam para ALTO RISCO:")
print(coefs.sort_values().head(12).round(3).to_string())

Termos que mais puxam para BAIXO RISCO:
da           0.418
quando       0.412
depois       0.408
quando eu    0.395
um           0.367
tempo        0.307
leve         0.281
um pouco     0.275
pouco        0.249
dolorido     0.246
doi          0.240
passou       0.234

Termos que mais puxam para ALTO RISCO:
nao           -0.511
consigo       -0.337
nao consigo   -0.337
ar            -0.335
para          -0.327
no peito      -0.311
junto         -0.264
no            -0.253
ha            -0.230
falta         -0.228
respirar      -0.225
agora         -0.224


**Esta é a saída mais reveladora do notebook, e a pior notícia.**

Olhe os termos que mais puxam para baixo risco: `da`, `quando`, `depois`, `quando eu`,
`um`. São palavras sem nenhum conteúdo clínico. O modelo percebeu que as frases de baixo
risco que **nós escrevemos** costumam ter estrutura condicional — "dói *quando* eu
aperto", "passou *depois* que descansei" — e está usando esse padrão de escrita como
atalho.

Ou seja: parte da acurácia vem de estilo de redação, não de medicina. Como as 40 frases
de cada classe saíram da mesma cabeça, cada classe ganhou um jeito de escrever próprio,
e o modelo achou esse jeito antes de achar o sintoma.

Do lado do alto risco aparece `nao` e `nao consigo` com peso forte. Aqui deu certo por
acaso ("não consigo respirar", "não consigo ficar em pé"), mas confirma a limitação da
negação: o modelo tratou "não" como sinal de gravidade. Numa frase como "não sinto dor
no peito", esse mesmo peso levaria à conclusão errada.

Corrigir isso exigiria frases escritas por pessoas diferentes, com estilos diferentes —
o que está fora do que dá para fazer nesta fase, mas precisa estar registrado.

### 12. Testando com frases novas

Frases que não estão na base, incluindo casos de fronteira escritos de propósito.

In [19]:
frases_novas = [
    "Estou com dor no peito há quarenta minutos e o braço esquerdo está dormente.",
    "Meu peito doeu um pouco quando eu levantei o sofá ontem.",
    "nao consigo respirar e meus labios estao roxos",
    "Fiquei com dor de cabeça leve depois de um dia longo de trabalho.",
    "Senti o coração acelerar depois do café da manhã, passou logo.",
    "Acordei sufocando de madrugada e precisei sentar para conseguir respirar.",
]

probs = modelo_lr.predict_proba(frases_novas)
classes = modelo_lr.named_steps["clf"].classes_
i_alto = list(classes).index("alto risco")

for f, p in zip(frases_novas, probs):
    print(f"{p[i_alto]:5.1%} alto risco  |  {f}")

59.4% alto risco  |  Estou com dor no peito há quarenta minutos e o braço esquerdo está dormente.


35.0% alto risco  |  Meu peito doeu um pouco quando eu levantei o sofá ontem.
64.3% alto risco  |  nao consigo respirar e meus labios estao roxos
44.4% alto risco  |  Fiquei com dor de cabeça leve depois de um dia longo de trabalho.
37.4% alto risco  |  Senti o coração acelerar depois do café da manhã, passou logo.
57.3% alto risco  |  Acordei sufocando de madrugada e precisei sentar para conseguir respirar.


**As probabilidades ficaram todas espremidas entre 35% e 65%.**

O caso mais preocupante: "nao consigo respirar e meus labios estao roxos" — cianose com
dificuldade respiratória, uma emergência clara — recebeu só 64% de alto risco. Deveria
estar perto de 100%.

O modelo acerta a direção, mas com pouca convicção. Isso é o tamanho da base aparecendo:
com 60 frases de treino não há evidência suficiente para o modelo se comprometer. Se
alguém usasse um corte de 70% para acionar atendimento prioritário, **esse paciente não
seria priorizado.**

É a demonstração prática de por que não se deve calibrar limiar clínico em base
sintética pequena.

### 13. Salvando os resultados

In [20]:
resultados = pd.DataFrame([
    {"modelo": "Regressão Logística", "acuracia_split_unico": accuracy_score(y_teste, pred_lr),
     "cv_media": cv_lr.mean(), "cv_desvio": cv_lr.std()},
    {"modelo": "Árvore de Decisão", "acuracia_split_unico": accuracy_score(y_teste, pred_dt),
     "cv_media": cv_dt.mean(), "cv_desvio": cv_dt.std()},
    {"modelo": "Baseline palavra-chave", "acuracia_split_unico": accuracy_score(y_teste, pred_baseline),
     "cv_media": np.nan, "cv_desvio": np.nan},
]).round(4)

resultados.to_csv(RAIZ / "dados" / "resultado_classificador.csv", index=False, encoding="utf-8")
resultados

,modelo,acuracia_split_unico,cv_media,cv_desvio
0,Regressão Logística,0.9,0.7750,0.0500
1,Árvore de Decisão,0.7,0.6375,0.0829
2,Baseline palavra-chave,0.6,NaN,NaN


### 14. Limitações, vieses e governança

O enunciado pede reflexão sobre qualidade e justiça dos dados. Segue o que eu
efetivamente encontrei, sem maquiar.

**1. A base é pequena e sintética.** 80 frases escritas pela própria equipe. Não há
paciente real, não há desfecho clínico, não há confirmação de diagnóstico. A acurácia
medida aqui não diz nada sobre desempenho no mundo real — diz apenas que o modelo
aprendeu a separar frases que nós mesmos escrevemos.

**2. Nós rotulamos, não um médico.** "Alto risco" e "baixo risco" são julgamento do
grupo, baseado em sinais de alarme clássicos. Alguns casos são discutíveis. Sem revisão
clínica, o gabarito é frágil — e um modelo nunca fica melhor que o próprio gabarito.

**3. Overfitting é risco concreto.** Mais de mil termos para 60 frases de treino: há
muito mais coluna que linha. O modelo tem espaço de sobra para decorar. A queda da
acurácia do split único para a validação cruzada é justamente esse efeito aparecendo.

**4. Viés de linguagem, o mais grave — e nós medimos.** A seção 11 mostrou que os
termos de maior peso para baixo risco são `quando`, `depois`, `da`, `um`: estrutura de
frase, não sintoma. Como as duas classes foram escritas pelas mesmas pessoas, cada uma
ganhou um estilo, e o modelo usou o estilo como atalho. A consequência real é séria:
quem tem pouca escolaridade, quem usa outro regionalismo, quem é idoso ou escreve com
erro de digitação não segue esse padrão de escrita, e seria classificado pelo jeito de
escrever em vez de pelo que sente. O erro cairia justamente sobre quem já tem menos
acesso a saúde.

**5. TF-IDF não entende negação — e há evidência disso.** A seção 11 mostra `nao` com
peso forte para alto risco, aprendido de "não consigo respirar". Numa frase como "não
sinto dor no peito", esse mesmo peso empurraria para a conclusão oposta à correta.

**6. Sem calibração, e as probabilidades são fracas.** A seção 12 mostrou um caso de
cianose com dificuldade respiratória recebendo apenas 64% de alto risco. As
probabilidades do `predict_proba` não foram calibradas e ficam todas perto de 50%, então
não servem para definir limiar de acionamento.

**Conclusão honesta:** este notebook demonstra o mecanismo de um triador automático. Ele
não é um triador. Usar isso com paciente de verdade, sem base real, sem rótulo clínico e
sem auditoria de viés, causaria dano.